# Intel Image Classification • Transfer Learning с ResNet50

Обучаем новый классификационный слой предобученной ResNet50. Backbone заморожен. Код повторяет учебную последовательность: Dataset → DataLoader → Model → Optimizer/Loss → Fit → метрики.

In [ ]:
# При необходимости установите зависимости из requirements.txt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms as T
from torchvision.models import resnet50, ResNet50_Weights
from pathlib import Path
import time, json
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import f1_score, confusion_matrix, classification_report

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_DIR = Path('data')
print('Устройство:', DEVICE)

## 1. Dataset и transforms

`ImageFolder` использует названия подпапок как классы. Для Intel dataset ожидаются `seg_train` и `seg_test`.

In [ ]:
classes = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
def find_split(split):
    for p in [DATA_DIR / split, DATA_DIR / split / split]:
        if p.is_dir() and all((p / c).is_dir() for c in classes): return p
    raise FileNotFoundError(f'Не найдена структура {split}/<class>')

train_transform = T.Compose([
    T.Resize((224, 224)), T.RandomHorizontalFlip(), T.RandomRotation(10),
    T.ToTensor(), T.Normalize([.485,.456,.406], [.229,.224,.225])
])
test_transform = T.Compose([
    T.Resize((224, 224)), T.ToTensor(),
    T.Normalize([.485,.456,.406], [.229,.224,.225])
])
train_dataset = datasets.ImageFolder(find_split('seg_train'), transform=train_transform)
test_dataset = datasets.ImageFolder(find_split('seg_test'), transform=test_transform)
print('Train:', len(train_dataset), 'Test:', len(test_dataset))
print('Классы:', train_dataset.class_to_idx)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
images, labels = next(iter(train_loader))
print('Изображения:', images.shape, 'Метки:', labels.shape)

## 2. Model — ResNet50

Загружаем ImageNet-веса, заменяем `fc`, замораживаем всю сеть и размораживаем только новый классификатор.

In [ ]:
weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights)
model.fc = nn.Linear(model.fc.in_features, len(train_dataset.classes))
for param in model.parameters(): param.requires_grad = False
for param in model.fc.parameters(): param.requires_grad = True
model = model.to(DEVICE)
print(model.fc)
print('Обучаемые параметры:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## 3. Optimizer, Loss и обучение

Backbone оставляем в `eval`, чтобы замороженные BatchNorm-слои не обновляли статистики.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
history = {'train_losses': [], 'train_accs': [], 'valid_losses': [], 'valid_accs': []}
start_time = time.perf_counter()
EPOCHS = 5

for epoch in range(EPOCHS):
    model.eval(); model.fc.train()
    loss_sum = correct = total = 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(); logits = model(x); loss = criterion(logits, y)
        loss.backward(); optimizer.step()
        loss_sum += loss.item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item(); total += y.size(0)
    history['train_losses'].append(loss_sum / total)
    history['train_accs'].append(correct / total)

    model.eval(); loss_sum = correct = total = 0
    with torch.inference_mode():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE); logits = model(x)
            loss_sum += criterion(logits, y).item() * y.size(0)
            correct += (logits.argmax(1) == y).sum().item(); total += y.size(0)
    history['valid_losses'].append(loss_sum / total)
    history['valid_accs'].append(correct / total)
    print(f"Epoch {epoch+1}/{EPOCHS} | train loss={history['train_losses'][-1]:.4f}, train acc={history['train_accs'][-1]:.4f} | test loss={history['valid_losses'][-1]:.4f}, test acc={history['valid_accs'][-1]:.4f}")

training_time = time.perf_counter() - start_time
print(f'Время обучения: {training_time:.1f} сек.')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history['train_losses'], label='train loss'); ax[0].plot(history['valid_losses'], label='test loss'); ax[0].set_title('Loss'); ax[0].legend(); ax[0].grid()
ax[1].plot(history['train_accs'], label='train accuracy'); ax[1].plot(history['valid_accs'], label='test accuracy'); ax[1].set_title('Accuracy'); ax[1].legend(); ax[1].grid()
plt.show()

## 4. F1-score и Confusion Matrix

In [ ]:
y_true, y_pred = [], []
model.eval()
with torch.inference_mode():
    for x, y in test_loader:
        pred = model(x.to(DEVICE)).argmax(1).cpu().numpy()
        y_pred.extend(pred.tolist()); y_true.extend(y.numpy().tolist())
f1 = f1_score(y_true, y_pred, average='macro')
cm = confusion_matrix(y_true, y_pred)
print('Macro F1:', round(f1, 4))
print(classification_report(y_true, y_pred, target_names=train_dataset.classes))
plt.figure(figsize=(8, 6)); plt.imshow(cm, cmap='Blues'); plt.colorbar()
plt.xticks(range(len(classes)), train_dataset.classes, rotation=45); plt.yticks(range(len(classes)), train_dataset.classes)
plt.xlabel('Предсказанный класс'); plt.ylabel('Истинный класс'); plt.title('Confusion matrix')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]): plt.text(j, i, cm[i,j], ha='center', va='center')
plt.tight_layout(); plt.show()

## 5. Сохранение модели

Streamlit-приложение использует эти веса и JSON с историей обучения и метриками.

In [ ]:
Path('artifacts').mkdir(exist_ok=True)
torch.save(model.cpu().state_dict(), 'artifacts/resnet50_intel.pth')
print('Веса сохранены: artifacts/resnet50_intel.pth')

# Для полной автоматизации сохранения метрик используйте train.py из проекта:
# python train.py --data_dir data --epochs 5 --batch_size 32